In [ ]:
# AIC 2026 - do toc do OCR tieng Viet tren keyframe that
SAMPLE_SIZE = 300
ZIP_URL = "<PRESIGNED_URL>"   # presigned, han 72h, chi doc dung mot file zip
WORK = "/tmp/aic"


In [ ]:
# Kaggle cap P100 (sm_60) hoac T4 (sm_75) tuy luc. torch cai san cua Kaggle
# chi build cho sm_70+, nen tren P100 moi phep tinh CUDA deu bao
# "no kernel image available". Phai quyet dinh TRUOC khi import torch,
# vi doi ban torch sau khi da import thi phai restart kernel.
import subprocess, os, sys, time, pathlib
os.makedirs(WORK, exist_ok=True)
def sh(c, quiet=False):
    r = subprocess.run(c, shell=True, capture_output=True, text=True)
    if not quiet: print("$", c, "\n", r.stdout[-1500:], flush=True)
    return r

gpu_name = sh("nvidia-smi --query-gpu=name --format=csv,noheader", quiet=True).stdout.strip()
print("GPU duoc cap:", gpu_name, flush=True)

sh("pip install -q easyocr 2>&1 | tail -3")

LEGACY = any(k in gpu_name for k in ("P100", "K80", "P4"))
if LEGACY:
    print(">>> GPU Pascal: ha torch ve 2.4.1+cu121 (con build sm_60)", flush=True)
    t0 = time.time()
    sh("pip install -q torch==2.4.1 torchvision==0.19.1 "
       "--index-url https://download.pytorch.org/whl/cu121 2>&1 | tail -5")
    print(f"cai xong sau {time.time()-t0:.0f}s", flush=True)


In [ ]:
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
USE_GPU = False
if torch.cuda.is_available():
    print("compute capability:", torch.cuda.get_device_capability(0))
    try:
        # Thu that mot phep tinh: cuda.is_available() van True tren P100 hong.
        x = torch.randn(8, 8, device="cuda") @ torch.randn(8, 8, device="cuda")
        torch.cuda.synchronize()
        USE_GPU = True
        print("phep tinh CUDA: OK")
    except Exception as e:
        print("phep tinh CUDA HONG ->", type(e).__name__, str(e)[:200])
print("=> chay tren", "GPU" if USE_GPU else "CPU")
if not USE_GPU:
    SAMPLE_SIZE = 40      # CPU rat cham, lay mau nho de van co so lieu
    print("   giam mau xuong", SAMPLE_SIZE)


In [ ]:
import zipfile, urllib.request
t0 = time.time(); zpath = f"{WORK}/kf.zip"
urllib.request.urlretrieve(ZIP_URL, zpath)
dl = time.time() - t0; mb = os.path.getsize(zpath)/1e6
print(f"tai {mb:.0f} MB / {dl:.0f}s = {mb/dl:.0f} MB/s", flush=True)

zf = zipfile.ZipFile(zpath)
members = [m for m in zf.namelist() if m.lower().endswith(".jpg")]
step = max(1, len(members)//SAMPLE_SIZE)
for m in members[::step][:SAMPLE_SIZE]:
    zf.extract(m, f"{WORK}/imgs")
paths = sorted(str(p) for p in pathlib.Path(f"{WORK}/imgs").rglob("*.jpg"))
from PIL import Image
print(f"anh trong zip {len(members)} | lay mau {len(paths)} | kich thuoc {Image.open(paths[0]).size}")


In [ ]:
import easyocr, numpy as np
reader = easyocr.Reader(["vi"], gpu=USE_GPU)
_ = reader.readtext(paths[0])                      # warmup roi moi bam gio

results, t0 = [], time.time()
for i, p in enumerate(paths):
    det = reader.readtext(p)
    results.append({"path": p.split("imgs/")[-1],
                    "texts": [d[1] for d in det],
                    "scores": [float(d[2]) for d in det]})
    if (i+1) % 25 == 0:
        print(f"  {i+1}/{len(paths)}  {(i+1)/(time.time()-t0):.2f} anh/s", flush=True)
elapsed = time.time() - t0
rate = len(paths)/elapsed
print(f"\nKET QUA: {len(paths)} anh / {elapsed:.0f}s = {rate:.2f} anh/s  ({'GPU' if USE_GPU else 'CPU'})")


In [ ]:
SHOT_LEVEL, FRAME_LEVEL = 87642, 177321
print(f"do duoc {rate:.2f} anh/s tren 1 {'GPU' if USE_GPU else 'CPU'}\n")
for name, n in [("muc shot ", SHOT_LEVEL), ("muc frame", FRAME_LEVEL)]:
    h = n/rate/3600
    print(f"{name} {n:7,d} anh -> {h:6.1f} h  (~{max(1, int(h/7)+1)} phien 9h, quota 30h/tuan)")
wt = sum(1 for r in results if r["texts"])
sc = [s for r in results for s in r["scores"]]
print(f"\nanh co chu : {wt}/{len(results)} = {wt/len(results):.1%}")
print(f"vung chu   : {len(sc)} tong, {len(sc)/max(1,len(results)):.1f}/anh")
if sc: print(f"tin cay    : trung vi {np.median(sc):.2f}")


In [ ]:
# Chat luong doc - phan quyet dinh OCR co dung duoc khong
shown = 0
for r in results:
    if len(r["texts"]) >= 2 and shown < 20:
        print(f"\n--- {r['path']}")
        for t, s in zip(r["texts"], r["scores"]):
            print(f"   [{s:.2f}] {t}")
        shown += 1


In [ ]:
import json
json.dump({"rate_img_per_sec": rate, "n": len(paths), "used_gpu": USE_GPU,
           "gpu_name": gpu_name, "results": results},
          open("/kaggle/working/ocr_benchmark.json","w",encoding="utf-8"),
          ensure_ascii=False, indent=1)
print("da luu /kaggle/working/ocr_benchmark.json")
